In [ ]:
pip install MFDFA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from MFDFA import MFDFA

# Base de datos: Polución

Lectura de base de datos

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Tesis/Data/pollutionOriginal.csv')

## 1) Variables endógenas o exógenas

Se imprime cada una de las variables del dataset

In [ ]:
#Se imprime el nombre de las columnas
print(df.columns)

Index(['No', 'year', 'month', 'day', 'hour', 'pm2.5', 'DEWP', 'TEMP', 'PRES',
       'cbwd', 'Iws', 'Is', 'Ir'],
      dtype='object')


Sabiendo que las variables exógenas son aquellas que se originan fuera del sistema, por lo cual en este caso estas serían las siguientes 12:

* No: Número
* year: Año
* month: mes
* day: día
* hour: hora
* DEWP: punto de rocío
* TEMP: temperatura
* PRES: presión
* cbdw: dirección del viento
* Iws: dirección del viento acumulada
* Is: horas de nieve acumulada
* Ir: horas de lluvia acumulada



Por su parte, como se busca predecir una variable en particular, se cuenta con una única variable endógena, es decir:

* pm 2.5: niveles de polución




## 2) Problema Univariante o multivariante

Como se apreció previamente, el problema involucra a múltiples variables en estudio, por lo tanto se clasifica al problema como **multivariante**.

## 3) Muestreo Regular o Irregular

In [ ]:
#Se copia el dataframe original
df2 = df.copy()

In [ ]:
# Se combinan columnas
df2['date'] = pd.to_datetime(df2[['year', 'month', 'day', 'hour']])

# Se establece la columna date como index
df2.set_index('date', inplace=False)

# Se eliminan las columnas originales junto con No, puesto que se tiene un nuevo index
df2.drop(['year', 'month', 'day', 'hour', 'No'], axis=1, inplace=True)

In [ ]:
# Calcular la diferencia entre tiempos consecutivos
df2['date_diff'] = df2['date'].diff()

# Revisar si todas las diferencias son iguales
regular = df2['date_diff'].iloc[1:].nunique() == 1

if regular:
    print("El muestreo es regular.")
else:
    print("El muestreo es irregular.")

El muestreo es regular.


## 4) Número de filas y columnas

Para encontrar el número de filas y columnas se utiliza el siguiente código.

In [ ]:
#Se imprime la cantidad de filas y columnas
print(df.shape)

(43824, 13)


En total se cuenta con **43824 filas** y **13 columnas**.

## 5) Serie estacionaria o no estacionaria




Para considerar o no si la serie es estacionaria, se aplica la prueba ADF a cada serie de tiempo por separado que compone el dataframe, y luego se revisa si existe una mayor cantidad de series estacionarias o no estacionarias. Se utiliza un nivel de significación de 5%.

Se considera que las hipótesis son las siguientes:



*   H0: La serie tiene raíz unitaria, por lo que se considera no estacionaria.
*   H1: La serie no tiene raíz unitaria, por lo que se considera estacionaria.

Cabe destacar que por el análisis EDA anterior, se deben reemplazar los valores faltantes de la columna "pm2.5", al igual que eliminar toda la columna "cbwd" por ser categórica.




In [ ]:
# Función para aplicar el test ADF y mostrar los resultados
def adf_test(series):
    result = adfuller(series, autolag='AIC')
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')


In [ ]:
#Se copia el dataframe original
df3 = df.copy()

In [ ]:
#Interpolación lineal para reemplazar datos faltantes
df3['pm2.5'] = df3['pm2.5'].interpolate(method='linear').bfill()

In [ ]:
#Se elimina cbwd al ser una variable categórica y no numérica
df3 = df3.drop(columns=["cbwd"])

In [ ]:
# Aplicar el test ADF a cada columna
for column in df3.columns:
    print(f'\nResultados del test ADF para la columna: {column}')
    adf_test(df3[column])


Resultados del test ADF para la columna: No


/usr/local/lib/python3.10/dist-packages/statsmodels/regression/linear_model.py:955: RuntimeWarning: divide by zero encountered in log
  llf = -nobs2*np.log(2*np.pi) - nobs2*np.log(ssr / nobs) - nobs2


ADF Statistic: 76.56745371846444
p-value: 1.0

Resultados del test ADF para la columna: year
ADF Statistic: -0.7072641780339606
p-value: 0.844858968744032

Resultados del test ADF para la columna: month
ADF Statistic: -3.3697680401930836
p-value: 0.012033677295235566

Resultados del test ADF para la columna: day
ADF Statistic: -13.508393509108474
p-value: 2.885213979773877e-25

Resultados del test ADF para la columna: hour
ADF Statistic: -794819558137323.2
p-value: 0.0

Resultados del test ADF para la columna: pm2.5
ADF Statistic: -21.347130264676284
p-value: 0.0

Resultados del test ADF para la columna: DEWP
ADF Statistic: -6.086701518331369
p-value: 1.0601628284120799e-07

Resultados del test ADF para la columna: TEMP
ADF Statistic: -3.8735275608584687
p-value: 0.0022385419293692223

Resultados del test ADF para la columna: PRES
ADF Statistic: -7.380073775883129
p-value: 8.518361949892657e-11

Resultados del test ADF para la columna: Iws
ADF Statistic: -33.923697114110375
p-value: 0.

Luego, se llega a los siguientes resultados considerando si es que se supera o no el nivel de significación:


* No: No Estacionaria
* year: No Estacionaria
* month: Estacionaria
* day: Estacionaria
* hour: Estacionaria
* DEWP: Estacionaria
* TEMP: Estacionaria
* PRES: Estacionaria
* cbdw: Estacionaria
* Iws: Estacionaria
* Is: Estacionaria
* Ir: Estacionaria
* pm 2.5: Estacionaria

Por lo anterior, se llega a que existe una mayor cantidad de series Estacionarias en el set de datos, por lo que, en general, el dataframe se considera como estacionario.

## 6) Complejidad

Para definir la complejidad del conjunto de series de tiempo, de acuerdo a lo propuesto en la metodología original, se utiliza el método MF-DFA. Luego, como el problema es multivariante, se calcula la complejidad para cada serie de tiempo y finalmente se promedia el resultado.

Se define la cantidad de lags de acuerdo con lo expuesto en la documentación de la biblioteca MF-DFA utilizada. En otras palabras, el límite inferior es equivalente a el orden usado + 1 (en este caso es 3), con el límite superior siendo aproximadamente la cuarta parte de la cantidad total de muestras (10000 aproximadamente). Siguiendo los ejemplos, los lags se generan logarítimicamente.

In [ ]:
# Se seleccionan lags
lag = np.unique(np.logspace(0.5, 4, 50).astype(int))

print(lag)

[    3     4     5     6     7     8    10    11    13    16    19    22
    26    31    37    43    51    61    71    84   100   117   138   163
   193   227   268   316   372   439   517   610   719   848  1000  1178
  1389  1637  1930  2275  2682  3162  3727  4393  5179  6105  7196  8483
 10000]


Siendo los exponentes fractales a utilizar, la lista de "q" se genera con valores entre 0 y 10 sin incluir al 0. Se podrían usar valores negativos, pero probando diferentes valores se llega a que lo más adecuado es utilizar valores positivos, de modo de evitar errores.

In [ ]:
# Se selecciona una lista de poderes q
q_list = np.linspace(0,10,15)
q_list = q_list[q_list!=0.0]

print(q_list)

[ 0.71428571  1.42857143  2.14285714  2.85714286  3.57142857  4.28571429
  5.          5.71428571  6.42857143  7.14285714  7.85714286  8.57142857
  9.28571429 10.        ]


Se prueban diferentes órdenes del polinomio a aproximar, llegando a que el más adecuado es 2.

In [ ]:
# Orden del ajuste polinomial
order = 2

Se realiza el análisis para cada serie de tiempo.

In [ ]:
# Diccionario para almacenar los resultados
resultados = {}
promedios = []
# Iterar sobre cada columna del DataFrame
for col in df3.columns:
    # Convertir la columna a un array de valores
    serie = df3[col].values

    # Aplicar el MF-DFA
    lag_values, dfa = MFDFA(serie, lag=lag, q=q_list, order=order)
    # Calcular los exponentes de Hurst (H)
    H_values = []
    for i, q in enumerate(q_list):
        H = np.polyfit(np.log(lag_values), np.log(dfa[:, i]), 1)[0]
        H_values.append(H)
    # Almacenar el valor promedio de H para la columna actual
    media_H = round(np.mean(H_values), 2)
    resultados[col] = media_H
    promedios.append(media_H)

# Imprimir los resultados
for col, H in resultados.items():
    print(f"El MFDFA para la columna {col} es: {H}")

# Calcular e imprimir el promedio de los promedios
promedio_global = round(np.mean(promedios), 2)
print(f"El promedio global de los promedios de MFDFA es: {promedio_global}")


El MFDFA para la columna No es: -0.05
El MFDFA para la columna year es: 2.19
El MFDFA para la columna month es: 2.33
El MFDFA para la columna day es: 2.22
El MFDFA para la columna hour es: 1.64
El MFDFA para la columna pm2.5 es: 0.91
El MFDFA para la columna DEWP es: 1.15
El MFDFA para la columna TEMP es: 1.1
El MFDFA para la columna PRES es: 1.26
El MFDFA para la columna Iws es: 0.87
El MFDFA para la columna Is es: 0.66
El MFDFA para la columna Ir es: 0.73
El promedio global de los promedios de MFDFA es: 1.25


En conclusión, el MFDFA alcanzado es de 1.25, que se puede interpretar como una muy baja complejidad. Sin contar las variables temporales, la complejidad cambia a baja, con un valor de 0.95.